# Comprehensive Analysis and Modeling Notebook

Welcome to this comprehensive notebook where we dive deep into the world of data analysis and machine learning. This document is meticulously crafted to guide you through various stages of data processing, modeling, and prediction. Here's what to expect:

## What This Notebook Offers:
1. **Data Preprocessing**: Initial steps to clean and prepare the data for analysis.
2. **Exploratory Data Analysis (EDA)**: Insights and patterns unraveled through visual and statistical methods.
3. **Feature Engineering**: Enhancing the dataset with new, informative features.
4. **Model Development**: Implementation of various machine learning models, including both traditional and advanced techniques.
5. **Evaluation and Optimization**: Assessing model performance and tuning them for better accuracy.
6. **Ensemble Techniques**: Leveraging the power of multiple models to improve predictions.
7. **Final Predictions and Submission**: Preparing the final predictions for submission, demonstrating the practical application of our analysis.

## Intended Audience:
This notebook is designed for both beginners and experienced practitioners in the field of data science. Whether you're looking to learn new skills, seeking to understand specific methodologies, or aiming to apply advanced techniques in machine learning, this notebook has something to offer.

## Feedback and Collaboration:
Your feedback is highly appreciated! If you have any suggestions, questions, or ideas for improvement, please feel free to share. Collaboration is the key to success in the ever-evolving field of data science, and your input is invaluable.

---

Let's embark on this data science journey together and uncover the stories hidden within the data!


In [ ]:
import sys
import gc

import pandas as pd
from sklearn.model_selection import StratifiedKFold
import numpy as np
from sklearn.metrics import roc_auc_score
import numpy as np
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.feature_extraction.text import TfidfVectorizer

from tokenizers import (
    decoders,
    models,
    normalizers,
    pre_tokenizers,
    processors,
    trainers,
    Tokenizer,
    SentencePieceBPETokenizer
)

from datasets import Dataset
from tqdm.auto import tqdm
from transformers import PreTrainedTokenizerFast

from sklearn.linear_model import SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import VotingClassifier

In [ ]:
import os
import pandas as pd

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    pass
else:
    sub = pd.read_csv('/kaggle/input/llm-detect-ai-generated-text/sample_submission.csv')
    sub.to_csv('submission.csv', index=False)
    sys.exit()

In [ ]:
test = pd.read_csv('/kaggle/input/llm-detect-ai-generated-text/test_essays.csv')
sub = pd.read_csv('/kaggle/input/llm-detect-ai-generated-text/sample_submission.csv')
train = pd.read_csv("/kaggle/input/daigt-v2-train-dataset/train_v2_drcat_02.csv", sep=',')

In [ ]:
train = train.drop_duplicates(subset=['text'])
train.reset_index(drop=True, inplace=True)

In [ ]:
LOWERCASE = False
VOCAB_SIZE = 42000

# Creating Byte-Pair Encoding Tokenizer

This cell initializes a Byte-Pair Encoding (BPE) tokenizer, a method effective for subword tokenization in NLP tasks. The tokenizer is configured with special tokens like `[UNK]`, `[PAD]`, `[CLS]`, `[SEP]`, and `[MASK]`. We use normalization and pre-tokenization strategies suitable for BPE. The tokenizer is trained on a subset of the dataset iteratively and wrapped in `PreTrainedTokenizerFast` for efficient tokenization. Finally, it's applied to both the test and training text data.


In [ ]:
# Creating Byte-Pair Encoding tokenizer
raw_tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
raw_tokenizer.normalizer = normalizers.Sequence([normalizers.NFC()] + [normalizers.Lowercase()] if LOWERCASE else [])
raw_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel()
special_tokens = ["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"]
trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=special_tokens)
dataset = Dataset.from_pandas(test[['text']])
def train_corp_iter(): 
    for i in range(0, len(dataset), 1000):
        yield dataset[i : i + 1000]["text"]
raw_tokenizer.train_from_iterator(train_corp_iter(), trainer=trainer)
tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=raw_tokenizer,
    unk_token="[UNK]",
    pad_token="[PAD]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]",
)
tokenized_texts_test = []

for text in tqdm(test['text'].tolist()):
    tokenized_texts_test.append(tokenizer.tokenize(text))

tokenized_texts_train = []

for text in tqdm(train['text'].tolist()):
    tokenized_texts_train.append(tokenizer.tokenize(text))

# TF-IDF Vectorization with Custom Tokenization

This cell implements the TF-IDF (Term Frequency-Inverse Document Frequency) vectorization process, customized for specific tokenization needs. The `TfidfVectorizer` is set up with a 3-5 n-gram range and various parameters including sublinear term frequency scaling and unicode accent stripping. Custom functions are used for both tokenization and preprocessing to maintain control over these processes. After fitting the vectorizer to the tokenized test data, we extract the vocabulary. This vocabulary is then used to initialize a new `TfidfVectorizer` which transforms both the training and test datasets. Post-processing, the vectorizer is deleted to free up memory.


In [ ]:
def dummy(text):
    return text
vectorizer = TfidfVectorizer(ngram_range=(3, 5), lowercase=False, sublinear_tf=True, analyzer = 'word',
    tokenizer = dummy,
    preprocessor = dummy,
    token_pattern = None, strip_accents='unicode')

vectorizer.fit(tokenized_texts_test)

# Getting vocab
vocab = vectorizer.vocabulary_

print(vocab)

vectorizer = TfidfVectorizer(ngram_range=(3, 5), lowercase=False, sublinear_tf=True, vocabulary=vocab,
                            analyzer = 'word',
                            tokenizer = dummy,
                            preprocessor = dummy,
                            token_pattern = None, strip_accents='unicode'
                            )

tf_train = vectorizer.fit_transform(tokenized_texts_train)
tf_test = vectorizer.transform(tokenized_texts_test)

del vectorizer
gc.collect()

In [ ]:
y_train = train['label'].values

# Ensemble Learning with Multiple Classifiers

This cell sets up an ensemble learning model using various classifiers, each with its specific configurations:

1. **Multinomial Naive Bayes**: 
   A `MultinomialNB` classifier with a smoothing parameter `alpha` set to 0.02.

2. **SGD Classifier**: 
   An `SGDClassifier` for linear models with a modified Huber loss function, a maximum of 8000 iterations, and a tolerance of 1e-4 for stopping criteria.

3. **LightGBM Classifier**: 
   An `LGBMClassifier` configured with custom parameters such as learning rate, lambda values, max depth, and more, specified in the `p6` dictionary.

4. **CatBoost Classifier**: 
   A `CatBoostClassifier` with 1000 iterations, silent mode (no verbose output), specific learning rate, L2 regularization, and a subsampling rate of 0.4.

5. **Ensemble Model - Voting Classifier**: 
   A `VotingClassifier` that combines the above models (`MultinomialNB`, `SGDClassifier`, `LGBMClassifier`, `CatBoostClassifier`) using soft voting. The weights for each classifier in the ensemble are specified, with a focus on the three non-Naive Bayes models.

The ensemble model is then trained on the transformed training data (`tf_train`) and labels (`y_train`). Finally, the ensemble model is used to predict probabilities on the test dataset (`tf_test`), and garbage collection is run to manage memory.
``


In [ ]:
clf = MultinomialNB(alpha=0.02)
sgd_model = SGDClassifier(max_iter=8000, tol=1e-4, loss="modified_huber") 
p6={'n_iter': 2500,'verbose': -1,'objective': 'cross_entropy','metric': 'auc',
    'learning_rate': 0.00581909898961407, 'colsample_bytree': 0.78,
    'colsample_bynode': 0.8, 'lambda_l1': 4.562963348932286, 
    'lambda_l2': 2.97485, 'min_data_in_leaf': 115, 'max_depth': 23, 'max_bin': 898}
lgb=LGBMClassifier(**p6)
cat=CatBoostClassifier(iterations=2500,
                       verbose=0,
                       l2_leaf_reg=6.6591278779517808,
                       learning_rate=0.005599066836106983,
                       subsample = 0.4,
                       allow_const_label=True,loss_function = 'CrossEntropy')
weights = [0.07,0.31,0.31,0.31]

ensemble = VotingClassifier(estimators=[('mnb',clf),
                                        ('sgd', sgd_model),
                                        ('lgb',lgb), 
                                        ('cat', cat)
                                       ],
                            weights=weights, voting='soft', n_jobs=-1)
ensemble.fit(tf_train, y_train)
gc.collect()
final_preds_bpe = ensemble.predict_proba(tf_test)[:,1]

del tokenized_texts_test, tokenized_texts_train, dataset, raw_tokenizer, tokenizer
_ = gc.collect()

# Integrating Sentencepiece Encoding with Machine Learning Models

## Tokenizer Setup
1. **Sentencepiece Tokenizer Initialization**:
   A `SentencePieceBPETokenizer` is initialized for subword tokenization.

2. **Normalization and Pre-Tokenization**:
   The tokenizer is configured with NFC normalization and optional lowercase conversion based on the `LOWERCASE` flag. Byte level pre-tokenization is also applied.

3. **Special Tokens and Training**:
   Special tokens like `[UNK]`, `[PAD]`, `[CLS]`, `[SEP]`, `[MASK]` are added. The tokenizer is trained on the test dataset using a custom generator function that iterates over the dataset in chunks.

4. **Tokenization**:
   The tokenizer processes both test and training datasets to create tokenized text data.

## TF-IDF Vectorization
1. **Vectorization Setup**:
   A `TfidfVectorizer` with a 3-5 n-gram range and custom tokenizer and preprocessor (`dummy` function) is used. The vectorizer is first fitted to the tokenized test data.

2. **Vocabulary Extraction and Transformation**:
   The vocabulary from the test data vectorization is extracted and used to initialize a new `TfidfVectorizer`. This vectorizer then transforms both training and test tokenized texts.

3. **Memory Management**:
   The vectorizer is deleted, and garbage collection is run to manage memory.

## Model Training and Prediction
1. **Model Initialization**:
   Multiple models including `MultinomialNB`, `SGDClassifier`, `LGBMClassifier`, and `CatBoostClassifier` are initialized with specific parameters.

2. **Ensemble Model Creation**:
   A `VotingClassifier` ensemble, using soft voting and specified weights, combines the aforementioned models.

3. **Training and Prediction**:
   The ensemble model is trained on the TF-IDF transformed training data and labels. It then predicts probabilities on the transformed test dataset.

4. **Final Predictions and Cleanup**:
   Predicted probabilities (`final_preds_spe`) are stored, and memory cleanup is performed with garbage collection.


In [ ]:
# # Creating Sentencepiece Encoding tokenizer
# raw_tokenizer = SentencePieceBPETokenizer()

# # Adding normalization and pre_tokenizer
# raw_tokenizer.normalizer = normalizers.Sequence([normalizers.NFC()] + [normalizers.Lowercase()] if LOWERCASE else [])
# raw_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel()
# # Adding special tokens and creating trainer instance
# special_tokens = ["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"]

# # Creating huggingface dataset object
# dataset = Dataset.from_pandas(test[['text']])

# def train_corp_iter():
#     """
#     A generator function for iterating over a dataset in chunks.
#     """    
#     for i in range(0, len(dataset), 300):
#         yield dataset[i : i + 300]["text"]

# # Training from iterator REMEMBER it's training on test set...
# raw_tokenizer.train_from_iterator(train_corp_iter())

# tokenizer = PreTrainedTokenizerFast(
#     tokenizer_object = raw_tokenizer,
#     unk_token="[UNK]",
#     pad_token="[PAD]",
#     cls_token="[CLS]",
#     sep_token="[SEP]",
#     mask_token="[MASK]",
# )

# tokenized_texts_test = []

# # Tokenize test set with new tokenizer
# for text in tqdm(test['text'].tolist()):
#     tokenized_texts_test.append(tokenizer.tokenize(text))

# # Tokenize train set
# tokenized_texts_train = []

# for text in tqdm(train['text'].tolist()):
#     tokenized_texts_train.append(tokenizer.tokenize(text))
    
# vectorizer = TfidfVectorizer(ngram_range=(3, 5), lowercase=False, sublinear_tf=True, analyzer = 'word',
#     tokenizer = dummy,
#     preprocessor = dummy,
#     token_pattern = None, strip_accents='unicode')

# vectorizer.fit(tokenized_texts_test)

# # Getting vocab
# vocab = vectorizer.vocabulary_

# print(vocab)

# vectorizer = TfidfVectorizer(ngram_range=(3, 5), lowercase=False, sublinear_tf=True, vocabulary=vocab,
#                             analyzer = 'word',
#                             tokenizer = dummy,
#                             preprocessor = dummy,
#                             token_pattern = None, strip_accents='unicode'
#                             )

# tf_train = vectorizer.fit_transform(tokenized_texts_train)
# tf_test = vectorizer.transform(tokenized_texts_test)

# del vectorizer
# gc.collect()

# y_train = train['label'].values

# def calculate_voting(tf_train, tf_test, y_train):
#     # Initialize classifiers
#     clf = MultinomialNB(alpha = 0.02)
#     sgd_model = SGDClassifier(max_iter=8000, tol=1e-4, loss="modified_huber") 
#     p7={'n_iter': 1500,'verbose': -1,'objective': 'cross_entropy','metric': 'auc',
#     'learning_rate': 0.0058, 'colsample_bytree': 0.78,
#     'colsample_bynode': 0.8, 'lambda_l1': 4.563, 
#     'lambda_l2': 2.97, 'min_data_in_leaf': 112, 'max_depth': 22, 'max_bin': 900}
#     lgb = LGBMClassifier(**p7)
#     cat=CatBoostClassifier(iterations=1000,
#                        verbose=0,
#                        l2_leaf_reg=6.659,
#                        learning_rate=0.0056,
#                        subsample = 0.4,
#                        allow_const_label=True,loss_function = 'CrossEntropy')
#     # Fit classifiers and make predictions
#     clf.fit(tf_train, y_train)
#     predictions_mnb = clf.predict_proba(tf_test)[:, 1]
#     del clf

#     sgd_model.fit(tf_train, y_train)
#     predictions_sgd = sgd_model.predict_proba(tf_test)[:, 1]
#     del sgd_model
    
#     lgb.fit(tf_train, y_train)
#     predictions_lgb = lgb.predict_proba(tf_test)[:, 1]
#     print('done with lightgbm')
#     del lgb
#     _ = gc.collect()
    
#     cat.fit(tf_train, y_train)
#     predictions_cat = cat.predict_proba(tf_test)[:, 1]
#     print('done with catboost')
#     del cat
    
#     # Define weights
#     weights = [0.07,0.31,0.31,0.31]

#     # Calculate weighted average of predictions
#     final_preds = (weights[0] * predictions_mnb + weights[1] * predictions_sgd + weights[2] * predictions_lgb + weights[3] * predictions_cat) / sum(weights)

#     # Garbage collection
#     gc.collect()

#     return final_preds
# final_preds_spe = calculate_voting(tf_train, tf_test, y_train)
# _ = gc.collect()
# ###############################

In [ ]:
# Create a boolean mask for values greater than 0.5
mask = final_preds_bpe > 0.5

# only to the values greater than 0.5
final_preds_bpe[mask] += 0.05

# Clip values to be between 0 and 1
final_preds_bpe = np.clip(final_preds_bpe, 0, 1)

# Final Submission and Closing Remarks

## Submission Preparation
In this final cell, we prepare our submission:

1. **Ensemble Prediction Averaging**:
   We combine the predictions from both the Byte-Pair Encoding (BPE) and Sentencepiece Encoding (SPE) models by averaging them. This approach helps in harnessing the strengths of both models.

   ```python
   sub['generated'] = (final_preds_bpe + final_preds_spe) / 2


In [ ]:
# sub['generated'] = 0.51 * np.array(final_preds_bpe) + 0.49 * final_preds_spe
sub['generated'] = final_preds_bpe
sub.to_csv('submission.csv', index=False)
sub